# NetraEdge - Face Recognition Training
## MobileFaceNet | GPU: T4 | Time: ~2 hours

In [ ]:
#@title Step 1: Install + GPU Check
!pip install -q onnx tqdm scikit-learn onnxscript
import torch
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name())
else:
    print('WARNING: No GPU! Runtime > Change runtime type > GPU')

In [ ]:
#@title Step 2: Load LFW Dataset
from sklearn.datasets import fetch_lfw_people
from PIL import Image
import numpy as np
import os

print('Loading LFW via sklearn...')
lfw = fetch_lfw_people(min_faces_per_person=20, resize=0.5, color=True)
print('Loaded:', lfw.images.shape[0], 'images,', len(lfw.target_names), 'identities')

output_dir = 'lfw_persons'
os.makedirs(output_dir, exist_ok=True)

for i, name in enumerate(lfw.target_names):
    mask = lfw.target == i
    if mask.sum() < 10:
        continue
    person_dir = os.path.join(output_dir, name.replace(' ', '_'))
    os.makedirs(person_dir, exist_ok=True)
    for j, idx in enumerate(np.where(mask)[0]):
        img = Image.fromarray(lfw.images[idx].astype(np.uint8))
        img = img.resize((112, 112), Image.BILINEAR)
        img.save(os.path.join(person_dir, str(j).zfill(4) + '.jpg'))

persons = [d for d in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, d))]
print('Saved', len(persons), 'persons to', output_dir)

In [ ]:
#@title Step 3: MobileFaceNet Architecture
import torch.nn as nn
import torch.nn.functional as F

class DWSep(nn.Module):
    def __init__(self, ic, oc, st=1):
        super().__init__()
        self.dw = nn.Conv2d(ic, ic, 3, st, 1, groups=ic, bias=False)
        self.b1 = nn.BatchNorm2d(ic)
        self.pw = nn.Conv2d(ic, oc, 1, bias=False)
        self.b2 = nn.BatchNorm2d(oc)
    def forward(self, x):
        return F.relu(self.b2(self.pw(F.relu(self.b1(self.dw(x))))))

class SE(nn.Module):
    def __init__(self, ch, r=4):
        super().__init__()
        m = max(ch // r, 8)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, m), nn.ReLU(True),
            nn.Linear(m, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.se(x).unsqueeze(-1).unsqueeze(-1)

class MB(nn.Module):
    def __init__(self, ic, oc, st=1):
        super().__init__()
        m = ic * 2
        self.ex = nn.Sequential(
            nn.Conv2d(ic, m, 1, bias=False), nn.BatchNorm2d(m), nn.ReLU(True))
        self.dw = DWSep(m, oc, st)
        self.se = SE(oc)
        self.res = (st == 1 and ic == oc)
    def forward(self, x):
        o = self.se(self.dw(self.ex(x)))
        return o + x if self.res else o

class MobileFaceNet(nn.Module):
    def __init__(self, ed=128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True))
        self.blk = nn.Sequential(
            MB(64, 64), MB(64, 128, 2), MB(128, 128),
            MB(128, 256, 2), MB(256, 256), MB(256, 256),
            MB(256, 512, 2), MB(512, 512), MB(512, 512))
        self.fin = nn.Sequential(
            nn.Conv2d(512, 512, 3, groups=512, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, ed, 1, bias=False), nn.BatchNorm2d(ed))
    def forward(self, x):
        x = self.fin(self.blk(self.stem(x)))
        return F.normalize(x.view(x.size(0), -1), p=2, dim=1)

model = MobileFaceNet(128)
nparams = sum(p.numel() for p in model.parameters())
print('MobileFaceNet params:', nparams)
print('Output shape:', model(torch.randn(2, 3, 112, 112)).shape)

In [ ]:
#@title Step 4: DataLoader
from torch.utils.data import Dataset, DataLoader
import random, glob

class LFWDataset(Dataset):
    def __init__(self, root='lfw_persons', max_identities=100, augment=True):
        self.samples = []
        self.augment = augment
        dirs = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
        dirs = dirs[:max_identities]
        for i, identity in enumerate(dirs):
            for p in glob.glob(os.path.join(root, identity, '*.jpg')):
                self.samples.append((p, i))
        print(len(self.samples), 'images,', len(dirs), 'identities')
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB').resize((112, 112))
        arr = np.array(img).astype(np.float32) / 255.0
        if self.augment:
            if random.random() > 0.5:
                arr = np.clip(arr + np.random.uniform(-0.12, 0.12), 0, 1)
            if random.random() > 0.5:
                arr = np.flip(arr, axis=1).copy()
        tensor = torch.from_numpy(arr).permute(2, 0, 1).float()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return (tensor - mean) / std, label

train_ds = LFWDataset('lfw_persons', 80, augment=True)
val_ds = LFWDataset('lfw_persons', 100, augment=False)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
print('Train:', len(train_ds), 'Val:', len(val_ds))

In [ ]:
#@title Step 5: Train 10 Epochs
import time
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MobileFaceNet(128).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
best_acc = 0

for epoch in range(10):
    t0 = time.time()
    model.train()
    train_correct = 0
    train_total = 0
    for images, labels in tqdm(train_loader, desc='Epoch ' + str(epoch + 1) + '/10'):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            val_correct += (model(images).argmax(1) == labels).sum().item()
            val_total += labels.size(0)
    train_acc = 100.0 * train_correct / train_total
    val_acc = 100.0 * val_correct / val_total
    scheduler.step()
    elapsed = int(time.time() - t0)
    print('Epoch ' + str(epoch + 1) + ': Train ' + str(round(train_acc, 1)) + '% Val ' + str(round(val_acc, 1)) + '% ' + str(elapsed) + 's')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'rec_best.pt')
        print('  Saved best model (' + str(round(val_acc, 1)) + '%)')
print('Training complete. Best: ' + str(round(best_acc, 1)) + '%')

In [ ]:
#@title Step 6: Export ONNX
model.cpu().eval()
torch.onnx.export(
    model, torch.randn(1, 3, 112, 112),
    'face_recognition.onnx',
    input_names=['input'],
    output_names=['output'],
    opset_version=13,
    dynamo=False)
sz = os.path.getsize('face_recognition.onnx') / 1e6
print('Exported: face_recognition.onnx (' + str(round(sz, 1)) + 'MB)')
print('Download and place in F:/PROJECTS/NetraEdge/models/')